Curator: Elizabeth (Liz) Osborn

Reviewer:  Louise Marie Maganto

Title: Analysis of a Stochastic Within-Host Model of Dengue Infection with Immune Response and Ornstein–Uhlenbeck Process

Pathogen: Dengue Fever (DF) DENV-1 to DENV-4

DOI: 10.1007/s00332-023-10004-4

Figure: 1

Outcome: Failed

Notes: x_bar (eq. 1.4) value not explicitly defined. Table 2 provides a value for b_bar, and the substitution x = ln(b) is stated on page 5 (eq. 1.3), 
which implies x_bar = ln(b_bar), but this relationship is never stated explicitly as a parameter definition.

In [7]:
from jax import numpy as jnp
variable_names = [
    'S',
    'I',
    'V',
    'Z',
    'x'
]
"""Names of the variables in the SDE model. The order of the variables should be the same as the order of the drift and diffusion terms returned by the drift_term and diffusion_term functions."""

parameter_names = [
    'mu',
    'alpha',
    'beta',
    'nu',
    'm',
    'gamma',
    'eta',
    'k',
    'theta',
    'delta',
    'rho',
    'x_bar',
    'sigma',
]
"""Names of the parameters in the SDE model. The order of the parameters should be the same as the order of the values returned by the drift_term and diffusion_term functions."""

initial_values = dict(
    S = 1000,       # Section 6 for System 1.4
    I = 1000,       # Section 6 for System 1.4
    V = 1100,       # Section 6 for System 1.4
    Z = 1100,       # Section 6 for System 1.4
    x = 2,           # Section 6 for System 1.4
    m = 5 #CHANGED: adding this to checkout my system, made this up
)
"""Dictionary of initial values for the variables in the SDE model. The keys should be the variable names in variable_names and the values should be the initial values for those variables."""

parameter_values = dict(
    mu = 80,        # Table 2 for system 1.4
    alpha = 0.2,    # Table 2 for system 1.4
    beta = 0.25,    # Table 2 for system 1.4
    nu = 0.001,     # Table 2 for system 1.4
    m = 20,         # Table 2 for system 1.4
    gamma = 0.8,    # Table 2 for system 1.4
    eta = 0.265,    # Table 2 for system 1.4
    k = 0.0624,     # Table 2 for system 1.4          
    theta = 0.0005, # Table 2 for system 1.4
    delta = 1/180,  # Table 2 for system 1.4         
    rho = 0.1,      # Table 2 for system 1.4 
    x_bar = None,   # Undefined - see notes
    sigma = 0.0316  # Table 2 for system 1.4
)
"""Dictionary of values for the parameters in the SDE model. The keys should be the parameter names in parameter_names and the values should be the values for those parameters."""

initial_time = 0.0
"""Initial time to simulate during testing and curation of the SDE model."""

final_time = 60000.0
"""Final time to simulate during testing and curation of the SDE model."""


def drift_term(t, y, p):
    """The drift term(s) of the SDE model

    Args:
        t: current time
        y: current values of the variables in the same order as variable_names
        p: current values of the parameters in the same order as parameter_names
    Returns:
        list: The drift term(s) of the SDE model in the same order as variable_names
    """
    s, i, v, z, x = y
    n = s + i + v + z
    mu, alpha, beta, nu, m, gamma, eta, k, theta, delta, rho, x_bar, sigma = p
    return [
        mu - alpha*s - jnp.exp(x)*s*v,
        jnp.exp(x)*s*v - beta*i - nu*i*z,
        m*i - gamma*v - jnp.exp(x)*s*v,
        eta + k*i + theta*i*z - delta*z,
        rho*(x_bar - x)
    ]

def diffusion_term(t, y, p):
    """The diffusion term(s) of the SDE model

    Args:
        t: current time
        y: current values of the variables in the same order as variable_names
        p: current values of the parameters in the same order as parameter_names
    Returns:
        list: The diffusion term(s) of the SDE model in the same order as variable_names
    """
    s, i, v, z, x = y
    n = s + i + v + z
    mu, alpha, beta, nu, m, gamma, eta, k, theta, delta, rho, x_bar, sigma = p
    return [
        0,
        0,
        0,
        0,
        sigma
        ]

# End Curation

# Begin Testing

*Do not modify anything below this cell.*

Successful implementations can execute the cells below in order without error to produce a figure.

## Do checks

In [ ]:
missing_ics = [n for n in variable_names if n not in initial_values]
missing_params = [n for n in parameter_names if n not in parameter_values]

found_errors = False
if len(missing_ics) > 0:
    print(f"Error: Missing initial values for variables: {missing_ics}")
    found_errors = True
if len(missing_params) > 0:
    print(f"Error: Missing values for parameters: {missing_params}")
    found_errors = True
test_drift = drift_term(initial_time, [initial_values[n] for n in variable_names], [parameter_values[n] for n in parameter_names])
test_diffusion = diffusion_term(initial_time, [initial_values[n] for n in variable_names], [parameter_values[n] for n in parameter_names])
if len(test_drift) != len(variable_names):
    print(f"Error: The drift term function should return a list of the same length as variable_names. Expected length {len(variable_names)}, but got {len(test_drift)}.")
    found_errors = True
if len(test_diffusion) != len(variable_names):
    print(f"Error: The diffusion term function should return a list of the same length as variable_names. Expected length {len(variable_names)}, but got {len(test_diffusion)}.")
    found_errors = True
if found_errors:
    raise ValueError("Failed to define the SDE model.")

TypeError: unsupported operand type(s) for -: 'NoneType' and 'int'

: 

## Do simulation test

In [ ]:
import diffrax
import jax
from jax import numpy as jnp
from matplotlib import pyplot as plt
import numpy as np

sim_times = np.linspace(initial_time, final_time, 100)
dt = (final_time - initial_time) / 1000
dr_term = diffrax.ODETerm(lambda t, y, p: jnp.array(drift_term(t, y, p)))
br_term = diffrax.VirtualBrownianTree(t0=initial_time, t1=final_time, tol=dt / 10, shape=(), key=jax.random.PRNGKey(0))
di_term = diffrax.ControlTerm(lambda t, y, p: jnp.array(diffusion_term(t, y, p)), br_term)
sde_terms = diffrax.MultiTerm(dr_term, di_term)
solver = diffrax.Euler()
solution = diffrax.diffeqsolve(
    sde_terms,
    solver,
    t0=initial_time,
    t1=final_time,
    dt0=dt,
    y0=jnp.asarray([initial_values[n] for n in variable_names]),
    args=jnp.asarray([parameter_values[n] for n in parameter_names]),
    saveat=diffrax.SaveAt(ts=jnp.asarray(sim_times)),
    max_steps=None,
    throw=True
).ys

fig, axs = plt.subplots(len(variable_names), 1, figsize=(10, 5 * len(variable_names)), layout="constrained")
for i, name in enumerate(variable_names):
    axs[i].plot(sim_times, solution[:, i])
    axs[i].set_title(name)

TracerArrayConversionError: The numpy.ndarray conversion method __array__() was called on traced array with shape float32[]
The error occurred while tracing the function _fn at c:\Users\Liz\anaconda3\envs\epi-sde\Lib\site-packages\equinox\_eval_shape.py:31 for jit. This concrete value was not available in Python because it depends on the value of the argument _dynamic[1][1].
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerArrayConversionError

In [ ]:
print('Sucessfully defined the SDE model and generated a test simulation plot.')